# Async Zarr cube slice → image display

Fast path: select `(time, frequency, polarization)`, extract one SKY plane on a **background thread**, show it with **matplotlib**.

Section **6** is the SkyWidget path from the FOV-cutout research: an async chunk read, **one** naive-SIN warp at the slice phase center, then zoom and slew sample that texture. A pan that leaves the cutout fades and reloads once.

Supporting code lives in this folder (`slice_async.py`, `fov_cutout.py`). It does **not** import `ovro_lwa_portal.viz` SourceReview helpers.

Research notes: `.agents/research-async-zarr-slice-display.md`, `.agents/research-accessor-async-hotpath.md`

In [ ]:
%matplotlib inline

from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

# Local helpers (separate from ovro_lwa_portal.viz)
_HERE = Path.cwd()
if (_HERE / "slice_async.py").is_file():
    sys.path.insert(0, str(_HERE))
elif (_HERE / "async_cube_slice" / "slice_async.py").is_file():
    sys.path.insert(0, str(_HERE / "async_cube_slice"))
else:
    # Running from repo root
    sys.path.insert(0, str(_HERE / "notebooks" / "async_cube_slice"))

from slice_async import (
    PlaneKey,
    extract_plane_async,
    extract_planes_parallel,
    make_demo_dataset,
    open_cube,
    robust_limits,
)

## 1. Open a cube

Set `ZARR_PATH` to a real store, or leave it `None` to use an in-memory demo cube.

In [ ]:
# Examples:
# ZARR_PATH = Path("/path/to/observation.zarr")
# ZARR_PATH = "s3://ovro/IV_1hMHz_Taper_shflux_coadd.zarr"
ZARR_PATH = "s3://ovro/IVsub_1hMHz_Taper_dewarped_aligned.zarr"
ZARR_STORAGE_OPTIONS = {"profile": "osn", "client_kwargs": {"endpoint_url":"https://caltech1.osn.mghpcc.org"}}

if ZARR_PATH is None:
    ds = make_demo_dataset()
    print("Using in-memory demo SKY cube")
else:
    ds = open_cube(
        ZARR_PATH,
        chunks="auto",
        lm_chunk=512,
        storage_options=ZARR_STORAGE_OPTIONS,
    )
    print(f"Opened {ZARR_PATH!r}")

print(ds)
print("sizes:", dict(ds.sizes))
print("SKY chunks:", getattr(ds["SKY"], "chunks", None))

## 2. Async extract of one plane + display

The executor runs `isel` + `.values` off the notebook UI thread. Display updates when the Future completes.

In [ ]:
TIME_IDX = 0
FREQ_IDX = 0
POL_IDX = 0
MAX_DISPLAY = 512  # stride so each side is ≤ this; None = full resolution

executor = ThreadPoolExecutor(max_workers=2)

fut = extract_plane_async(
    ds,
    TIME_IDX,
    FREQ_IDX,
    POL_IDX,
    executor=executor,
    max_size=MAX_DISPLAY,
)
result = fut.result()  # wait; in a UI you would attach a done callback instead

vmin, vmax = robust_limits(result.data)
fig, ax = plt.subplots(figsize=(6, 5))
# Transpose for imshow so horizontal ≈ m, vertical ≈ l (matches radport.plot)
im = ax.imshow(
    result.data.T,
    origin="lower",
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    aspect="equal",
)
ax.set_xlabel("m (display)")
ax.set_ylabel("l (display)")
ax.set_title(
    f"SKY t={result.key.time_idx} f={result.key.freq_idx} "
    f"p={result.key.pol_idx}  ({result.elapsed_s * 1e3:.1f} ms)"
)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()
print(f"shape={result.data.shape}, dtype={result.data.dtype}, elapsed={result.elapsed_s:.3f}s")

## 3. Interactive selectors (ipywidgets)

Changing a slider submits a new extract; the previous Future is ignored via a generation counter.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

n_time = int(ds.sizes["time"])
n_freq = int(ds.sizes["frequency"])
n_pol = int(ds.sizes["polarization"])

time_w = widgets.IntSlider(value=0, min=0, max=max(0, n_time - 1), description="time")
freq_w = widgets.IntSlider(value=0, min=0, max=max(0, n_freq - 1), description="freq")
pol_w = widgets.IntSlider(value=0, min=0, max=max(0, n_pol - 1), description="pol")
status = widgets.HTML(value="Ready")
out = widgets.Output()

_generation = 0


def _schedule_main(callback):
    """Run UI updates on the IPython io_loop (not the worker thread)."""
    try:
        from IPython import get_ipython

        loop = get_ipython().kernel.io_loop
        loop.add_callback(callback)
    except Exception:
        callback()


def _request_plane(_change=None):
    global _generation
    _generation += 1
    gen = _generation
    t, f, p = int(time_w.value), int(freq_w.value), int(pol_w.value)
    status.value = f"Loading t={t} f={f} p={p}…"

    fut = extract_plane_async(
        ds, t, f, p, executor=executor, max_size=MAX_DISPLAY
    )

    def _on_done(f):
        if gen != _generation:
            return  # superseded
        res = f.result()

        def _draw():
            if gen != _generation:
                return
            vmin, vmax = robust_limits(res.data)
            with out:
                out.clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(6, 5))
                ax.imshow(
                    res.data.T,
                    origin="lower",
                    cmap="viridis",
                    vmin=vmin,
                    vmax=vmax,
                    aspect="equal",
                )
                ax.set_title(
                    f"SKY t={res.key.time_idx} f={res.key.freq_idx} "
                    f"p={res.key.pol_idx}  ({res.elapsed_s * 1e3:.1f} ms)"
                )
                plt.show()
            status.value = (
                f"Loaded shape={res.data.shape} in {res.elapsed_s * 1e3:.1f} ms"
            )

        _schedule_main(_draw)

    fut.add_done_callback(_on_done)


for w in (time_w, freq_w, pol_w):
    w.observe(_request_plane, names="value")

display(widgets.VBox([time_w, freq_w, pol_w, status, out]))
_request_plane()  # first frame


## 4. Parallel extract of many planes

Same helper scaled with `ThreadPoolExecutor` — useful when prefetching a time sequence or a small grid of frequencies.

In [ ]:
keys = [
    PlaneKey(time_idx=t, freq_idx=0, pol_idx=1)
    for t in range(min(4, n_time))
]

results = extract_planes_parallel(ds, keys, max_workers=4, max_size=MAX_DISPLAY)

ncols = len(results)
fig, axes = plt.subplots(1, ncols, figsize=(3.2 * ncols, 3.2), squeeze=False)
for ax, res in zip(axes[0], results):
    vmin, vmax = robust_limits(res.data)
    ax.imshow(res.data.T, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(f"t={res.key.time_idx}\n{res.elapsed_s * 1e3:.0f} ms")
    ax.set_xticks([])
    ax.set_yticks([])
plt.suptitle("Parallel plane extracts")
plt.tight_layout()
plt.show()

print("elapsed (ms):", [round(r.elapsed_s * 1e3, 1) for r in results])

## 6. SkyWidget — one native-SIN warp per cutout

Sections 2–4 stride a full plane into matplotlib. This cell loads a FOV-sized native cutout on the same executor and warps it **once** onto naive SIN at the slice phase center.

Zoom and a slew that still fits that cutout leave the texture in place (the shader samples it). A pan that no longer fits fades the overlay, then one async read builds a new cutout and warps that once. Run this **before** the cleanup cell, and create the widget in this cell (do not display it from another cell).

In [ ]:
import zarr
import ipywidgets as widgets
from IPython.display import display
from astropy.coordinates import SkyCoord
from astrowidget import SkyWidget

from fov_cutout import (
    FovOverlaySession,
    attach_fov_pan_reload,
    celestial_wcs_for_time,
)

FOV_DEG = 8.0


def _schedule_main(callback):
    """Run widget updates on the IPython io_loop."""
    try:
        from IPython import get_ipython

        get_ipython().kernel.io_loop.add_callback(callback)
    except Exception:
        callback()


def _open_sky_zarr(path, storage_options):
    """Consolidated SKY array for chunk-aligned reads."""
    text = str(path)
    if text.startswith("s3://"):
        import s3fs

        fs = s3fs.S3FileSystem(**(storage_options or {}))
        mapper = s3fs.S3Map(root=text.removeprefix("s3://"), s3=fs, check=False)
        return zarr.open_consolidated(mapper, mode="r")["SKY"]
    return zarr.open_consolidated(text, mode="r")["SKY"]


if ZARR_PATH is None:
    print("Demo cube has no per-time WCS; set ZARR_PATH to use the SkyWidget.")
else:
    sky_arr = _open_sky_zarr(ZARR_PATH, ZARR_STORAGE_OPTIONS)
    wcs0 = celestial_wcs_for_time(ds, TIME_IDX, freq_idx=FREQ_IDX)
    if wcs0 is None:
        raise RuntimeError(f"No WCS at t={TIME_IDX} f={FREQ_IDX}")
    phase = SkyCoord(
        float(wcs0.wcs.crval[0]),
        float(wcs0.wcs.crval[1]),
        unit="deg",
        frame="fk5",
    )
    sky = SkyWidget()
    _lines: list[str] = []
    log_html = widgets.HTML()

    def _log(msg: str) -> None:
        _lines.append(msg)
        log_html.value = "<br>".join(_lines[-8:])

    session = FovOverlaySession(
        sky,
        ds,
        sky_arr,
        pol_idx=POL_IDX,
        executor=executor,
        schedule_main=_schedule_main,
        log=_log,
        debounce_s=0.3,
        fade_opacity=0.35,
    )
    session.set_slice_indices(TIME_IDX, FREQ_IDX)
    attach_fov_pan_reload(sky, session.on_pan)
    session.push_slice(phase, FOV_DEG, t=TIME_IDX, f=FREQ_IDX, reason="open")
    display(
        widgets.VBox(
            [
                widgets.HTML(
                    "<b>SkyWidget</b> — one native-SIN warp at the phase center. "
                    "Zoom stays put. A pan off the cutout fades, then reloads."
                ),
                sky,
                log_html,
            ]
        )
    )


## 5. Cleanup

In [ ]:
executor.shutdown(wait=False)
print("executor shut down")